# Demo resuelto: auditar antes de interpretar

**Clave del facilitador.** Los datos son sintéticos y el objetivo es hacer visible una pérdida no aleatoria de observaciones.

In [ ]:
from pathlib import Path
import pandas as pd

CANDIDATES = [
    Path.cwd(),
    Path.cwd() / 'course/exercises/data_audit_demo',
    Path.cwd().parent,
]
ROOT = next(
    (p for p in CANDIDATES if (p / 'data/estudiantes.csv').exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError('Abre la carpeta data_audit_demo o la raíz del repositorio')
DATA = ROOT / 'data'

In [ ]:
estudiantes = pd.read_csv(DATA / 'estudiantes.csv')
hogares = pd.read_csv(DATA / 'caracteristicas_hogar.csv')

print(f'Estudiantes antes del merge: {len(estudiantes):,}')
print(f'Hogares en tabla auxiliar: {len(hogares):,}')

## 1. Conservar todas las observaciones durante la auditoría

In [ ]:
auditoria = estudiantes.merge(
    hogares,
    on='hogar_id',
    how='left',
    indicator=True,
    validate='many_to_one',
)

conteo_merge = auditoria['_merge'].value_counts(dropna=False)
conteo_merge

In [ ]:
auditoria['en_tabla_auxiliar'] = auditoria['_merge'].eq('both')

tasas_zona = (
    auditoria.groupby('zona')['en_tabla_auxiliar']
    .agg(['count', 'mean'])
    .rename(columns={'count': 'estudiantes', 'mean': 'tasa_correspondencia'})
)
tasas_zona['tasa_correspondencia'] = tasas_zona['tasa_correspondencia'].round(3)
tasas_zona

In [ ]:
tasas_distrito = (
    auditoria.groupby('distrito')['en_tabla_auxiliar']
    .agg(['count', 'mean'])
    .rename(columns={'count': 'estudiantes', 'mean': 'tasa_correspondencia'})
    .sort_values('tasa_correspondencia')
)
tasas_distrito['tasa_correspondencia'] = tasas_distrito['tasa_correspondencia'].round(3)
tasas_distrito

## 2. Convertir el diagnóstico en comprobaciones

In [ ]:
assert len(auditoria) == len(estudiantes), 'El merge cambió el número de estudiantes'
assert auditoria['estudiante_id'].is_unique, 'Se duplicaron estudiantes'
assert auditoria['en_tabla_auxiliar'].mean() >= 0.70, 'Cobertura auxiliar demasiado baja'

print('Comprobaciones superadas')

## 3. Responder la pregunta con la población adecuada

La característica auxiliar no es necesaria para calcular el descriptivo principal. Por eso usamos la base completa de estudiantes y reportamos la cobertura auxiliar por separado.

In [ ]:
tabla_auditada = (
    estudiantes.groupby('juntos', as_index=False)
    .agg(
        estudiantes=('estudiante_id', 'size'),
        asistencia_media=('asiste_regularmente', 'mean'),
    )
)
tabla_auditada['asistencia_media'] = tabla_auditada['asistencia_media'].round(3)
tabla_auditada

In [ ]:
muestra_completa = estudiantes[['estudiante_id', 'juntos', 'asiste_regularmente']]
muestra_reducida = auditoria.loc[auditoria['en_tabla_auxiliar']]

comparacion = pd.DataFrame({
    'muestra': ['completa', 'inner merge'],
    'n': [len(muestra_completa), len(muestra_reducida)],
    'brecha_descriptiva': [
        muestra_completa.groupby('juntos')['asiste_regularmente'].mean().diff().iloc[-1],
        muestra_reducida.groupby('juntos')['asiste_regularmente'].mean().diff().iloc[-1],
    ],
})
comparacion['brecha_descriptiva'] = comparacion['brecha_descriptiva'].round(3)
comparacion

## 4. Interpretación

La tabla auditada describe diferencias en estos datos sintéticos. No demuestra que Juntos cause cambios en asistencia: participación y asistencia pueden estar relacionadas con ruralidad, selección, territorio y otras características no observadas.